# Running InterviewPlayground

This notebook walks through the core InterviewPlayground workflow:
loading a preset, interviewing simulated participants, and scoring
the resulting transcripts with InterviewReportCard.

**Requirements:** an API key from a model provider(described in [README](../README.md#setup)) — loading the preset itself makes
no LLM calls, but `participant.ask()` and `study.evaluate()` do.

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."

from interviewplayground import load_preset

## Load a preset study

In [ ]:
study = load_preset("obesity_weight_management")

print(f"{len(study.insights)} insights")
print(f"{len(study.research_questions)} research questions")
print(f"{len(study.participants)} participants")

## Inspect a participant

In [ ]:
p = study.participants[0]
print(p.persona)
print()
print("Traits:", p.knowledge, p.verbosity, p.memory, p.reflexivity, p.disclosure, p.understanding)

## Interview the participant

`ask()` retrieves the participant's most relevant memories and
generates a response grounded in their persona and traits.

In [ ]:
response = p.ask("How have weight conversations with your doctor gone?")
print(response)

## Use the interview guide

Every preset includes `research_questions` and an `interview_guide` —
the same briefing a human interviewer would get. See the README section
["Using the interview guide with your own AI interviewer"](../README.md#using-the-interview-guide-with-your-own-ai-interviewer)
for how to turn this into instructions for a real AI interviewer.

In [ ]:
for topic in study.interview_guide:
    print(topic["topic"])
    for subtopic in topic["subtopics"]:
        print(f"  - {subtopic}")

## Interview a few more participants

`study.evaluate()` scores whatever is in each participant's
`transcript`, so interview a handful of participants with the same
questions before evaluating.

In [ ]:
questions = [
    "How have weight conversations with your doctor gone?",
    "What, if anything, has a doctor suggested for managing your weight?",
    "How did that conversation make you feel?",
]

for p in study.participants[:3]:
    for q in questions:
        p.ask(q)

## Evaluate the interview

`study.evaluate()` runs the full InterviewReportCard suite: one
descriptive bucket (`conversation_length`) plus three LLM-judged
dimensions (`participant_responses`, `interviewer_behavior`,
`participant_experience`).

In [ ]:
results = study.evaluate()
for dimension, metrics in results.items():
    print(dimension)
    for k, v in metrics.items():
        if k != "per_participant":
            print(f"  {k}: {v}")